# 4.3 Classification Models — Assignment with Answers


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)
print('Ready.')

## Q1 — Logistic Regression

In [ ]:
for C in [0.01, 1, 100]:
    m=LogisticRegression(C=C,max_iter=1000,random_state=42).fit(X_tr_s,y_tr)
    print(f'C={C}: accuracy={accuracy_score(y_te,m.predict(X_te_s)):.4f}')
print('C=0.01 (heavy regularization) → simpler model')
print('C=100 (light regularization) → can overfit on small datasets but here data is clean')

**Why it works:** C is the inverse of regularization strength. Small C → strong penalty → simpler model. Large C → weak penalty → fits training data more closely. Breast cancer data is well-structured so all three perform well.

## Q2 — Random Forest

In [ ]:
for n in [50,100,200]:
    rf=RandomForestClassifier(n_estimators=n,random_state=42).fit(X_tr,y_tr)
    print(f'n={n}: accuracy={accuracy_score(y_te,rf.predict(X_te)):.4f}')
rf=RandomForestClassifier(n_estimators=100,random_state=42).fit(X_tr,y_tr)
top5=pd.Series(rf.feature_importances_,index=data.feature_names).nlargest(5)
print('\nTop 5 features:'); display(top5.to_frame('importance').round(4))

**Why it works:** More trees improve performance up to a point (diminishing returns). Feature importances in RF = mean decrease in Gini impurity — features that split data cleanly at the top of many trees get higher importance.

## Q3 — Gradient Boosting

In [ ]:
for lr in [0.01,0.1,1.0]:
    gb=GradientBoostingClassifier(n_estimators=100,learning_rate=lr,max_depth=3,random_state=42).fit(X_tr,y_tr)
    print(f'lr={lr}: accuracy={accuracy_score(y_te,gb.predict(X_te)):.4f}')
print('\nlr=1.0 often overfits — each tree overcorrects the previous errors')

**Why it works:** Learning rate scales the contribution of each tree. High rate → overfit (trees overcorrect). Low rate → need more trees but generalizes better. lr=0.1 with n=100 is a commonly reliable starting point.

## Q4 — Model Comparison

In [ ]:
results={}
results['Logistic']=accuracy_score(y_te,LogisticRegression(max_iter=1000).fit(X_tr_s,y_tr).predict(X_te_s))
results['KNN']=accuracy_score(y_te,KNeighborsClassifier(5).fit(X_tr_s,y_tr).predict(X_te_s))
results['NaiveBayes']=accuracy_score(y_te,GaussianNB().fit(X_tr_s,y_tr).predict(X_te_s))
results['DecTree']=accuracy_score(y_te,DecisionTreeClassifier(max_depth=5,random_state=42).fit(X_tr,y_tr).predict(X_te))
results['RandForest']=accuracy_score(y_te,RandomForestClassifier(100,random_state=42).fit(X_tr,y_tr).predict(X_te))
results['GradBoost']=accuracy_score(y_te,GradientBoostingClassifier(100,random_state=42).fit(X_tr,y_tr).predict(X_te))
df=pd.Series(results).sort_values().to_frame('accuracy')
display(df)
df.plot.barh(figsize=(8,4),title='Model Accuracy Comparison');plt.tight_layout();plt.show()